# Fine-tune Impleo

Adapt all model parameters using masked panel reconstruction. Hidden target genes are not used in the fine-tuning loss. Use a new output directory for each run.

In [ ]:
from pathlib import Path
import sys
import subprocess
import numpy as np
import torch

ROOT = Path.cwd()
if ROOT.name == "examples":
    ROOT = ROOT.parent
if not (ROOT / "impleo").is_dir():
    raise RuntimeError("Open this notebook from the Impleo repository")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ASSETS = ROOT / "data" / "assets"
EXAMPLE = ASSETS / "starmap_p128_v0"
if not (EXAMPLE / "input_panel.npy").exists():
    subprocess.run([sys.executable, "-m", "data.download", "--packs", "example",
                    "--out", str(ASSETS)], cwd=ROOT, check=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(6)
from impleo.inference import read_names
panel = read_names(EXAMPLE / "protocol.json", "panel_genes")
targets = read_names(EXAMPLE / "hidden_genes.json")
values = np.load(EXAMPLE / "input_panel.npy", allow_pickle=False)
truth = np.load(EXAMPLE / "truth.npy", allow_pickle=False)
CHECKPOINT = ROOT / "checkpoints" / "impleo.pt"
print(f"{len(values)} cells; {len(panel)} panel genes; {len(targets)} target genes; {DEVICE}")

In [ ]:
from training.finetune import finetune, make_splits
splits = make_splits(len(values))
print({name: len(rows) for name, rows in splits.items()})
FT_DIR = ROOT / "results" / "finetuning"
model, ck = finetune(CHECKPOINT, values, panel, FT_DIR,
                     splits=splits, epochs=5, seed=42, device=DEVICE)
print("Saved:", FT_DIR / "best.pt")

In [ ]:
from impleo.inference import predict
from evaluation.metrics import gene_pearson
prediction = predict(model, ck, values, panel, targets)
print("Mean gene Pearson (all cells):", np.nanmean(gene_pearson(prediction, truth)))